 1. 카드 데이터 로드 -> 일단 체크카드만
 2. 카드 → 검색용 텍스트 변환
 3. 페르소나 정의
 6. Vector DB 생성 / 로드
 7. RAG 검색
 8. 프롬프트 (Top 3 강제)

In [ ]:
from openai import OpenAI
from getpass import getpass
import json
import os

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 1. 카드 데이터 로드
# ==================================================
def load_card_data_from_folder(folder_path: str) -> list:
    cards = []
    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                cards.append(json.load(f))
    return cards


CARD_DATA_DIR = "../data/check_cards_benefits"
card_data_list = load_card_data_from_folder(CARD_DATA_DIR)

if not card_data_list:
    raise ValueError("카드 JSON 데이터가 없습니다.")


# ==================================================
# 2. 카드 → 검색용 텍스트 변환
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type', '정보없음')}
혜택: {card.get('benefit')}
전월실적: {card.get('performance')}
"""


texts = [card_to_text(card) for card in card_data_list]


# ==================================================
# 3. 페르소나 정의 (Fallback 포함)
# ==================================================
PERSONAS = {
    "20대 학습·생활형": {
        "description": "대중교통, 카페, 소액 결제가 잦고 실적 부담 없는 소비 패턴",
        "keywords": ["버스", "지하철", "대중교통", "카페", "편의점"]
    },
    "30대 직장인·활동형": {
        "description": "직장 생활 중심, 이동·활동 소비 비중이 높은 소비 패턴",
        "keywords": ["주유", "차", "출장", "회사"]
    },
    "40대 가족·생활형": {
        "description": "생활비·고정 지출 중심 소비 패턴",
        "keywords": ["마트", "공과금", "자녀", "학원"]
    },
    "기타 / 복합 소비형": {
        "description": "명확한 페르소나로 분류되지 않는 복합 소비 패턴",
        "keywords": []
    }
}


# ==================================================
# 4. 페르소나 자동 추론
# ==================================================
def infer_persona(text: str) -> dict:
    scores = {}

    for name, persona in PERSONAS.items():
        scores[name] = sum(1 for k in persona["keywords"] if k in text)

    max_score = max(scores.values())
    if max_score == 0:
        return PERSONAS["기타 / 복합 소비형"]

    return PERSONAS[max(scores, key=scores.get)]


# ==================================================
# 5. OpenAI API Key
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 6. Vector DB 생성 / 로드
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

VECTOR_DB_DIR = "./card_vector_db"

if os.path.exists(VECTOR_DB_DIR):
    vectordb = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=embedding
    )
else:
    vectordb = Chroma.from_texts(
        texts=texts,
        embedding=embedding,
        persist_directory=VECTOR_DB_DIR
    )
    vectordb.persist()


# ==================================================
# 7. RAG 검색
# ==================================================
def retrieve_cards(query: str, k: int = 5) -> str:
    docs = vectordb.similarity_search(query, k=k)
    return "\n\n".join(doc.page_content for doc in docs)


# ==================================================
# 8. 프롬프트 (Top 3 강제)
# ==================================================
def build_prompt(age: str, card_type: str, question: str, persona_desc: str, retrieved_cards: str) -> str:
    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 [카드 정보]에 포함된 내용만 사용하여
사용자의 요청에 맞는 카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 카드 유형: {card_type}

[사용자 질문]
{question}

[판단 기준]
- 사용자 질문과 가장 잘 맞는 카드 3개를 선택하세요.
- 카드 정보에 없는 혜택은 절대 생성하지 마세요.
- 카드 유형이 다를 경우 추천하지 마세요.

[출력 형식]
아래 형식을 반드시 그대로 사용하세요.

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 근거:

2️⃣ 카드명:
- 카드사:
- 추천 근거:

3️⃣ 카드명:
- 카드사:
- 추천 근거:

[사용자 소비 패턴 참고]
{persona_desc}

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 9. 질문 처리
# ==================================================
def recommend_cards(age: str, card_type: str, user_question: str) -> str:
    persona = infer_persona(user_question)
    retrieved_cards = retrieve_cards(f"{user_question} {card_type}")

    system_prompt = build_prompt(
        age=age,
        card_type=card_type,
        question=user_question,
        persona_desc=persona["description"],
        retrieved_cards=retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": system_prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 10. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 Top 3 카드 추천 챗봇\n")

    age = input("1️⃣ 나이를 입력하세요: ")
    card_type = input("2️⃣ 카드 유형을 선택하세요 (신용카드 / 체크카드): ")
    question = input("3️⃣ 어떤 카드를 원하시나요? : ")

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    result = recommend_cards(age, card_type, question)
    print(result)

    print("\n==============================")


💳 카드 추천 챗봇


🤖 카드 추천 결과
TOP 3 카드 추천

1️⃣ 카드명: 락스타 체크카드
- 카드사: KB국민카드
- 추천 이유: 대중교통 10% 청구할인 혜택이 있어 교통 이용 시 부담을 줄여줍니다.

2️⃣ 카드명: 총무 체크카드
- 카드사: KB국민카드
- 추천 이유: 대중교통 5% 청구할인 혜택이 있어 교통 이용 시 추가 혜택을 받을 수 있습니다.

3️⃣ 카드명: 토스유스카드(USS)
- 카드사: 토스
- 추천 이유: 교통카드 사용 가능한 카드로, 교통 이용 시 편리하게 이용할 수 있습니다.



# 신용카드 + 체크카드 둘 다 적용

In [ ]:
from openai import OpenAI
from getpass import getpass
import json
import os

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 1. 카드 데이터 로드 (카드 유형 포함)
# ==================================================
def load_card_data_from_folder(folder_path: str, card_type: str) -> list:
    cards = []
    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                card = json.load(f)
                card["card_type"] = card_type   # 강제 주입
                cards.append(card)
    return cards


# 현재는 체크카드만 존재
CHECK_CARD_DIR = "../data/check_cards_benefits"
CREDIT_CARD_DIR = "../data/credit_cards_benefits"  # 추후 업로드 예정

card_data_list = []
card_data_list.extend(load_card_data_from_folder(CHECK_CARD_DIR, "체크카드"))

# 신용카드 폴더가 생기면 자동 로드
if os.path.exists(CREDIT_CARD_DIR):
    card_data_list.extend(load_card_data_from_folder(CREDIT_CARD_DIR, "신용카드"))

if not card_data_list:
    raise ValueError("카드 JSON 데이터가 없습니다.")


# ==================================================
# 2. 카드 → 검색용 텍스트 변환
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type')}
혜택: {card.get('benefit')}
전월실적: {card.get('performance')}
"""


texts = [card_to_text(card) for card in card_data_list]


# ==================================================
# 3. 페르소나 정의 (Fallback 포함)
# ==================================================
PERSONAS = {
    "20대 학습·생활형": {
        "description": "대중교통·카페·소액 결제가 잦은 소비 패턴",
        "keywords": ["버스", "지하철", "대중교통", "카페", "편의점"]
    },
    "30대 직장인·활동형": {
        "description": "직장 생활 중심, 이동·활동 소비 비중이 높은 패턴",
        "keywords": ["주유", "차", "출장", "회사"]
    },
    "40대 가족·생활형": {
        "description": "생활비·고정 지출 중심 소비 패턴",
        "keywords": ["마트", "공과금", "자녀", "학원"]
    },
    "기타 / 복합 소비형": {
        "description": "특정 페르소나로 분류되지 않는 복합 소비 패턴",
        "keywords": []
    }
}


def infer_persona(text: str) -> dict:
    scores = {name: sum(1 for k in p["keywords"] if k in text)
              for name, p in PERSONAS.items()}
    if max(scores.values()) == 0:
        return PERSONAS["기타 / 복합 소비형"]
    return PERSONAS[max(scores, key=scores.get)]


# ==================================================
# 4. OpenAI API Key
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 5. Vector DB 생성 / 로드
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

VECTOR_DB_DIR = "./card_vector_db"

if os.path.exists(VECTOR_DB_DIR):
    vectordb = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=embedding
    )
else:
    vectordb = Chroma.from_texts(
        texts=texts,
        embedding=embedding,
        persist_directory=VECTOR_DB_DIR
    )
    vectordb.persist()


# ==================================================
# 6. 카드 유형 필터링 RAG 검색
# ==================================================
def retrieve_cards(query: str, card_type: str, k: int = 5) -> str:
    docs = vectordb.similarity_search(query, k=k * 2)

    filtered = [
        doc.page_content
        for doc in docs
        if f"카드유형: {card_type}" in doc.page_content
    ]

    if not filtered:
        return ""

    return "\n\n".join(filtered[:k])


# ==================================================
# 7. 프롬프트 (Top 3 강제)
# ==================================================
def build_prompt(age, card_type, question, persona_desc, retrieved_cards):
    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 [카드 정보]에 포함된 내용만 사용하여
사용자의 요청에 맞는 카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 카드 유형: {card_type}

[사용자 질문]
{question}

[규칙]
- 카드 정보에 없는 혜택은 절대 생성하지 마세요.
- 카드 유형이 다른 카드는 추천하지 마세요.

[출력 형식]

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 근거:

2️⃣ 카드명:
- 카드사:
- 추천 근거:

3️⃣ 카드명:
- 카드사:
- 추천 근거:

[사용자 소비 패턴 참고]
{persona_desc}

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 8. 추천 실행
# ==================================================
def recommend_cards(age, card_type, question):
    persona = infer_persona(question)
    retrieved_cards = retrieve_cards(question, card_type)

    if not retrieved_cards:
        return f"""
선택하신 카드 유형({card_type})에 대한
추천 가능한 카드 데이터가 아직 없습니다.

현재는 체크카드만 추천이 가능하며,
신용카드 데이터는 추후 제공될 예정입니다.
"""

    prompt = build_prompt(
        age, card_type, question,
        persona["description"],
        retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 9. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 카드 유형 분기 기반 Top 3 카드 추천 챗봇\n")

    age = input("1️⃣ 나이: ")
    card_type = input("2️⃣ 카드 유형 선택 (신용카드 / 체크카드): ")
    question = input("3️⃣ 어떤 카드를 원하시나요? : ")

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    result = recommend_cards(age, card_type, question)
    print(result)

    print("\n==============================")


## (카드 유형 자동 판별 + 미선택 대응 포함)

In [18]:
from openai import OpenAI
from getpass import getpass
import json
import os

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 1. 카드 데이터 로드 (체크카드만)
# ==================================================
def load_card_data_from_folder(folder_path: str) -> list:
    cards = []
    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                card = json.load(f)
                card["card_type"] = "체크카드"
                cards.append(card)
    return cards


CARD_DATA_DIR = "../data/check_cards_benefits"
card_data_list = load_card_data_from_folder(CARD_DATA_DIR)

if not card_data_list:
    raise ValueError("카드 JSON 데이터가 없습니다.")


# ==================================================
# 2. 카드 → 검색용 텍스트
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type')}
혜택: {card.get('benefit')}
전월실적: {card.get('performance')}
"""


texts = [card_to_text(card) for card in card_data_list]


# ==================================================
# 3. OpenAI API
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 4. Vector DB 생성 / 로드
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

VECTOR_DB_DIR = "./card_vector_db"

if os.path.exists(VECTOR_DB_DIR):
    vectordb = Chroma(
        persist_directory=VECTOR_DB_DIR,
        embedding_function=embedding
    )
else:
    vectordb = Chroma.from_texts(
        texts=texts,
        embedding=embedding,
        persist_directory=VECTOR_DB_DIR
    )
    vectordb.persist()


# ==================================================
# 5. 자연어 조건 → 긍정 / 부정 분리 (규칙 기반)
# ==================================================
def split_conditions(text: str):
    negative_hints = ["안", "않", "별로", "거의", "잘 안", "싫"]
    tokens = text.replace(",", " ").split()

    positive = []
    negative = []

    for i, token in enumerate(tokens):
        window = " ".join(tokens[max(0, i-2):i+1])
        if any(neg in window for neg in negative_hints):
            negative.append(token)
        else:
            positive.append(token)

    # 중복 제거
    return " ".join(set(positive)), " ".join(set(negative))


# ==================================================
# 6. RAG 검색
# ==================================================
def retrieve_cards(query: str, k: int = 5) -> str:
    docs = vectordb.similarity_search(query, k=k)
    return "\n\n".join(doc.page_content for doc in docs)


# ==================================================
# 7. 프롬프트 (자연어 조건 반영)
# ==================================================
def build_prompt(age, card_type, condition_text, positive, negative, retrieved_cards):
    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 [카드 정보]에 포함된 내용만 사용하여
사용자의 생활 패턴과 조건에 가장 잘 맞는 카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 희망 카드 유형: {card_type}

[사용자 생활 패턴 설명]
{condition_text}

[중요하게 반영해야 할 조건]
{positive}

[추천 시 배제해야 할 영역]
{negative if negative else "없음"}

[지시 사항]
1. 중요한 조건과 직접적으로 관련된 혜택이 명시된 카드만 우선 고려하세요.
2. 배제 영역과 관련된 혜택은 추천 근거로 사용하지 마세요.
3. 카드 정보에 없는 혜택, 할인율, 조건은 절대 생성하지 마세요.
4. 할인 혜택이 크다고 판단할 경우, 반드시 카드 혜택 문구를 근거로 설명하세요.

[출력 형식]
아래 형식을 반드시 그대로 사용하세요.

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 이유:

2️⃣ 카드명:
- 카드사:
- 추천 이유:

3️⃣ 카드명:
- 카드사:
- 추천 이유:

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 8. 추천 실행
# ==================================================
def recommend_cards(age, card_type, condition_text):
    positive, negative = split_conditions(condition_text)

    search_query = f"""
    사용자가 중요하게 생각하는 조건: {positive}
    사용자가 피하고 싶은 영역: {negative}
    카드 유형: {card_type}
    """

    retrieved_cards = retrieve_cards(search_query)

    system_prompt = build_prompt(
        age=age,
        card_type=card_type,
        condition_text=condition_text,
        positive=positive,
        negative=negative,
        retrieved_cards=retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": system_prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 9. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 생활 패턴 기반 카드 추천 챗봇\n")

    age = input("1️⃣ 나이를 입력하세요: ")
    card_type = input("2️⃣ 어떤 카드를 찾고 계신가요? (신용카드 / 체크카드): ")

    condition = input(
        "3️⃣ 생활 패턴과 원하는 조건을 자유롭게 적어주세요:\n"
        "   예) 버스를 많이 타지만 지하철은 잘 안 타요. 카페는 자주 가요.\n👉 "
    )

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    result = recommend_cards(age, card_type, condition)
    print(result)

    print("\n==============================")


💳 생활 패턴 기반 카드 추천 챗봇


🤖 카드 추천 결과
TOP 3 카드 추천

1️⃣ 카드명:
- VIVA+ 체크카드
- 카드사: 하나카드
- 추천 이유: 카페를 많이 이용하시는 분께는 Lunch Time 7% 캐쉬백 서비스가 매우 유용할 수 있습니다.

2️⃣ 카드명:
- 총무 체크카드
- 카드사: KB국민카드
- 추천 이유: 카페 이용 시 커피음료전문점 업종 5% 환급할인이 제공되며, 대중교통(버스, 지하철) 5% 청구할인 혜택도 받을 수 있습니다.

3️⃣ 카드명:
- 일상의 기쁨 Dream 체크카드
- 카드사: IBK기업은행
- 추천 이유: 버스 이용 시 이용건당 100원 할인 혜택과 주요 커피전문점 10% 할인 혜택이 제공되어 카페와 대중교통을 자주 이용하시는 분께 적합할 수 있습니다.



# 신용카드(주석) + 체크카드

In [22]:
from openai import OpenAI
from getpass import getpass
import json
import os

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 1. 카드 데이터 로드 함수
# ==================================================
def load_card_data_from_folder(folder_path: str, card_type: str) -> list:
    cards = []
    if not os.path.exists(folder_path):
        return cards

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                card = json.load(f)
                card["card_type"] = card_type
                cards.append(card)
    return cards


# ==================================================
# 2. 카드 데이터 로드
# ==================================================
CHECK_CARD_DIR = "../data/check_cards_benefits"
CREDIT_CARD_DIR = "../data/credit_cards_benefits"  # ❌ 아직 데이터 없음

check_cards = load_card_data_from_folder(CHECK_CARD_DIR, "체크카드")

# ❌ 신용카드 데이터 (추후 사용 예정)
# credit_cards = load_card_data_from_folder(CREDIT_CARD_DIR, "신용카드")

if not check_cards:
    raise ValueError("체크카드 JSON 데이터가 없습니다.")


# ==================================================
# 3. 카드 → 검색용 텍스트 변환
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type')}
혜택: {card.get('benefit')}
전월실적: {card.get('performance')}
"""


check_texts = [card_to_text(card) for card in check_cards]


# ==================================================
# 4. OpenAI API
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 5. Vector DB 생성 / 로드
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

CHECK_VECTOR_DB_DIR = "./check_card_vector_db"

if os.path.exists(CHECK_VECTOR_DB_DIR):
    check_vectordb = Chroma(
        persist_directory=CHECK_VECTOR_DB_DIR,
        embedding_function=embedding
    )
else:
    check_vectordb = Chroma.from_texts(
        texts=check_texts,
        embedding=embedding,
        persist_directory=CHECK_VECTOR_DB_DIR
    )
    check_vectordb.persist()


# ==================================================
# 6. 자연어 조건 → 긍정 / 부정 분리
# ==================================================
def split_conditions(text: str):
    negative_hints = ["안", "않", "별로", "거의", "잘 안", "싫"]
    tokens = text.replace(",", " ").split()

    positive, negative = [], []

    for i, token in enumerate(tokens):
        window = " ".join(tokens[max(0, i-2):i+1])
        if any(neg in window for neg in negative_hints):
            negative.append(token)
        else:
            positive.append(token)

    return " ".join(set(positive)), " ".join(set(negative))


# ==================================================
# 7. RAG 검색 (카드 유형별)
# ==================================================
def retrieve_cards(query: str, card_type: str, k: int = 5) -> str:
    if card_type == "체크카드":
        docs = check_vectordb.similarity_search(query, k=k)
    else:
        docs = []

    return "\n\n".join(doc.page_content for doc in docs)


# ==================================================
# 8. 프롬프트 (요구사항 요약 포함)
# ==================================================
def build_prompt(age, card_type, condition_text, positive, negative, retrieved_cards):
    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 [카드 정보]에 포함된 내용만 사용하여
사용자의 생활 패턴과 조건에 가장 잘 맞는 카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 희망 카드 유형: {card_type}

[사용자 원문 입력]
{condition_text}

[중요 조건]
{positive}

[배제 조건]
{negative if negative else "없음"}

[출력 형식]
아래 형식을 반드시 그대로 사용하세요.

[입력한 요구사항 요약]
- 주요 사용 조건:
- 제외 조건:
- 카드 유형:

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 이유: (요구사항 요약 → 카드 혜택 → 적합성 결론 순서로 설명)

2️⃣ 카드명:
- 카드사:
- 추천 이유:

3️⃣ 카드명:
- 카드사:
- 추천 이유:

[규칙]
- 카드 정보에 없는 혜택, 할인율, 조건은 절대 생성하지 마세요.
- 모든 추천 이유는 카드 혜택 문구를 근거로 작성하세요.

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 9. 추천 실행
# ==================================================
def recommend_cards(age, card_type, condition_text):
    positive, negative = split_conditions(condition_text)

    search_query = f"""
    중요 조건: {positive}
    배제 조건: {negative}
    """

    retrieved_cards = retrieve_cards(search_query, card_type)

    if card_type == "신용카드" and not retrieved_cards:
        return (
            "⚠️ 현재 신용카드 데이터가 준비되지 않아 추천이 어렵습니다.\n"
            "체크카드 기준으로 다시 시도해 주세요."
        )

    system_prompt = build_prompt(
        age=age,
        card_type=card_type,
        condition_text=condition_text,
        positive=positive,
        negative=negative,
        retrieved_cards=retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": system_prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 10. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 생활 패턴 기반 카드 추천 챗봇\n")

    age = input("1️⃣ 나이를 입력하세요: ")
    card_type = input("2️⃣ 어떤 카드를 찾고 계신가요? (신용카드 / 체크카드): ")

    condition = input(
        "3️⃣ 생활 패턴과 원하는 조건을 자유롭게 적어주세요:\n"
        "   예) 버스를 많이 타지만 지하철은 잘 안 타요. 카페는 자주 가요.\n👉 "
    )

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    result = recommend_cards(age, card_type, condition)
    print(result)

    print("\n==============================")


💳 생활 패턴 기반 카드 추천 챗봇


🤖 카드 추천 결과
[입력한 요구사항 요약]
- 주요 사용 조건: 카페를 많이 가는 편이며 캐시백 및 할인 혜택을 원함
- 제외 조건: 없음
- 카드 유형: 체크카드

TOP 3 카드 추천

1️⃣ 카드명: 롯데마트 라서 즐거운 체크카드
- 카드사: 우리카드
- 추천 이유: 카페를 많이 가시는 분들을 위한 스타벅스, 투썸플레이스 20% 캐시백 할인 및 모든 영화관 3,000원 캐시백 할인이 제공되어 매우 적합합니다.

2️⃣ 카드명: 현대백화점 체크카드
- 카드사: iM뱅크
- 추천 이유: 카페 이용 시 스타벅스, 커피빈, 엔제리너스 10% 결제일 캐시백 혜택이 있어 적합합니다.

3️⃣ 카드명: 우리성당 체크카드
- 카드사: 우리카드
- 추천 이유: 카페 이용 시 스타벅스, 투썸플레이스 20% 할인 혜택이 제공되어 적합합니다.



# 한 번 더 물어보는 로직

In [ ]:
from openai import OpenAI
from getpass import getpass
import json
import os
import shutil

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 0. Vector DB 초기화 (잠금 오류 방지)
# ==================================================
VECTOR_DB_DIR = "./check_card_vector_db"

if os.path.exists(VECTOR_DB_DIR):
    try:
        shutil.rmtree(VECTOR_DB_DIR)
    except PermissionError:
        print("⚠️ Vector DB가 사용 중입니다. 모든 Python 프로세스를 종료 후 다시 실행하세요.")
        exit()


# ==================================================
# 1. 카드 데이터 로드 (체크카드만)
# ==================================================
def load_card_data_from_folder(folder_path: str) -> list:
    cards = []
    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                card = json.load(f)
                card["card_type"] = "체크카드"
                cards.append(card)
    return cards


CARD_DATA_DIR = "../data/check_cards_benefits"
cards = load_card_data_from_folder(CARD_DATA_DIR)

if not cards:
    raise ValueError("체크카드 JSON 데이터가 없습니다.")


# ==================================================
# 2. 카드 → 검색용 텍스트
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type')}
혜택: {card.get('benefit')}
전월실적: {card.get('performance')}
캐시백: {card.get('cashback')}
"""


texts = [card_to_text(card) for card in cards]


# ==================================================
# 3. OpenAI API
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 4. Vector DB 생성
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

vectordb = Chroma.from_texts(
    texts=texts,
    embedding=embedding,
    persist_directory=VECTOR_DB_DIR
)
vectordb.persist()


# ==================================================
# 5. 자연어 조건 분해
# ==================================================
def split_conditions(text: str):
    negative_hints = ["안", "않", "별로", "잘 안", "싫"]
    tokens = text.replace(",", " ").split()

    positive, negative = [], []

    for i, token in enumerate(tokens):
        window = " ".join(tokens[max(0, i-2):i+1])
        if any(n in window for n in negative_hints):
            negative.append(token)
        else:
            positive.append(token)

    return list(set(positive)), list(set(negative))


# ==================================================
# 6. 요구사항 충족 여부 판단
# ==================================================
def analyze_coverage(requirements, retrieved_text):
    coverage = {}
    for req in requirements:
        coverage[req] = req in retrieved_text
    return coverage


# ==================================================
# 7. 추가 질문 (트레이드오프)
# ==================================================
def ask_priority():
    print("\n⚖️ 모든 조건을 동시에 만족하는 카드는 없습니다.")
    print("어떤 조건을 더 중요하게 생각하시나요?")
    print("1️⃣ 캐시백")
    print("2️⃣ 카페 할인")

    choice = input("번호 선택: ")
    return "캐시백" if choice == "1" else "카페"


# ==================================================
# 8. RAG 검색
# ==================================================
def retrieve_cards(query: str, k: int = 5) -> str:
    docs = vectordb.similarity_search(query, k=k)
    return "\n\n".join(doc.page_content for doc in docs)


# ==================================================
# 9. 프롬프트
# ==================================================
def build_prompt(age, condition_text, positive, negative, coverage, retrieved_cards):
    coverage_text = "\n".join(
        [f"- {k}: {'충족됨' if v else '충족 불가'}" for k, v in coverage.items()]
    )

    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 카드 정보에 포함된 내용만 사용하여
사용자 조건에 가장 적합한 체크카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 카드 유형: 체크카드

[사용자 생활 패턴]
{condition_text}

[중요 조건]
{", ".join(positive)}

[배제 조건]
{", ".join(negative) if negative else "없음"}

[요구사항 충족 분석]
{coverage_text}

[지시사항]
- 모든 조건을 만족하는 카드가 없다면,
  어떤 조건이 충족되었고 어떤 조건이 충족되지 않았는지
  반드시 추천 이유에 포함해 설명하세요.
- 카드 정보에 없는 혜택은 절대 생성하지 마세요.

[출력 형식]

입력한 요구사항 요약:
- 주요 조건:
- 배제 조건:

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 이유:

2️⃣ 카드명:
- 카드사:
- 추천 이유:

3️⃣ 카드명:
- 카드사:
- 추천 이유:

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 10. 추천 실행
# ==================================================
def recommend_cards(age, condition_text):
    positive, negative = split_conditions(condition_text)

    retrieved_cards = retrieve_cards(condition_text)
    coverage = analyze_coverage(positive, retrieved_cards)

    # 트레이드오프 발생 시 추가 질문
    if "캐시백" in positive and "카페" in positive:
        if not coverage.get("캐시백", True):
            priority = ask_priority()
            retrieved_cards = retrieve_cards(f"{condition_text} {priority}")

    system_prompt = build_prompt(
        age=age,
        condition_text=condition_text,
        positive=positive,
        negative=negative,
        coverage=coverage,
        retrieved_cards=retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": system_prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 11. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 생활 패턴 기반 카드 추천 챗봇\n")

    age = input("1️⃣ 나이를 입력하세요: ")
    condition = input(
        "2️⃣ 생활 패턴과 원하는 조건을 자유롭게 적어주세요:\n"
        "   예) 카페는 많이 가지만 영화는 잘 안 봐요. 캐시백도 중요해요.\n👉 "
    )

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    result = recommend_cards(age, condition)
    print(result)

    print("\n==============================")


⚠️ Vector DB가 사용 중입니다. 모든 Python 프로세스를 종료 후 다시 실행하세요.
💳 생활 패턴 기반 카드 추천 챗봇


🤖 카드 추천 결과
입력한 요구사항 요약:
- 주요 조건: 신용
- 배제 조건: 없음

TOP 3 카드 추천

1️⃣ 카드명: 신한카드 SOL글로벌 체크
- 카드사: 신한카드
- 추천 이유: 대중교통 및 이동통신요금에 대한 10% 캐시백 혜택이 있으며, 전월실적이 없어도 신청 가능합니다.

2️⃣ 카드명: K-패스 신한카드 체크
- 카드사: 신한카드
- 추천 이유: 대중교통 이용 시 10% 결제일 할인 혜택이 있습니다.

3️⃣ 카드명: 신한카드 Hey Young 체크
- 카드사: 신한카드
- 추천 이유: 대중교통 및 다양한 일상 서비스에 대한 캐시백 혜택이 있습니다.



: 

In [ ]:
from openai import OpenAI
from getpass import getpass
import json
import os
import shutil

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


# ==================================================
# 0. Vector DB 초기화 (Windows 잠금 방지)
# ==================================================
CHECK_VECTOR_DB_DIR = "./check_card_vector_db"
# CREDIT_VECTOR_DB_DIR = "./credit_card_vector_db"  # 🔜 신용카드용

if os.path.exists(CHECK_VECTOR_DB_DIR):
    try:
        shutil.rmtree(CHECK_VECTOR_DB_DIR)
    except PermissionError:
        print("⚠️ Vector DB가 사용 중입니다. Python 프로세스 종료 후 재실행하세요.")
        exit()

# if os.path.exists(CREDIT_VECTOR_DB_DIR):
#     shutil.rmtree(CREDIT_VECTOR_DB_DIR)


# ==================================================
# 1. 카드 데이터 로드
# ==================================================
def load_card_data(folder_path: str, card_type: str) -> list:
    cards = []
    if not os.path.exists(folder_path):
        return cards

    for file in os.listdir(folder_path):
        if file.endswith(".json"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                card = json.load(f)
                card["card_type"] = card_type
                cards.append(card)
    return cards


CHECK_CARD_DIR = "../data/check_cards_benefits"
# CREDIT_CARD_DIR = "../data/credit_cards_benefits"  # 🔜 준비되면 사용

check_cards = load_card_data(CHECK_CARD_DIR, "체크카드")
# credit_cards = load_card_data(CREDIT_CARD_DIR, "신용카드")

if not check_cards:
    raise ValueError("체크카드 JSON 데이터가 없습니다.")


# ==================================================
# 2. 카드 → 검색용 텍스트
# ==================================================
def card_to_text(card: dict) -> str:
    return f"""
카드명: {card.get('card_name')}
카드사: {card.get('company')}
카드유형: {card.get('card_type')}
혜택: {card.get('benefit')}
캐시백: {card.get('cashback')}
전월실적: {card.get('performance')}
"""


check_texts = [card_to_text(card) for card in check_cards]
# credit_texts = [card_to_text(card) for card in credit_cards]


# ==================================================
# 3. OpenAI API
# ==================================================
MY_API_KEY = getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)


# ==================================================
# 4. Vector DB 생성
# ==================================================
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=MY_API_KEY
)

check_vectordb = Chroma.from_texts(
    texts=check_texts,
    embedding=embedding,
    persist_directory=CHECK_VECTOR_DB_DIR
)
check_vectordb.persist()

# 🔜 신용카드 DB
# credit_vectordb = Chroma.from_texts(
#     texts=credit_texts,
#     embedding=embedding,
#     persist_directory=CREDIT_VECTOR_DB_DIR
# )
# credit_vectordb.persist()


# ==================================================
# 5. 자연어 조건 분리
# ==================================================
def split_conditions(text: str):
    negative_hints = ["안", "않", "별로", "잘 안", "싫"]
    tokens = text.replace(",", " ").split()

    positive, negative = [], []

    for i, token in enumerate(tokens):
        window = " ".join(tokens[max(0, i-2):i+1])
        if any(n in window for n in negative_hints):
            negative.append(token)
        else:
            positive.append(token)

    return list(set(positive)), list(set(negative))


# ==================================================
# 6. 요구사항 충족 여부 분석
# ==================================================
def analyze_coverage(requirements, retrieved_text):
    return {req: req in retrieved_text for req in requirements}


# ==================================================
# 7. 추가 질문 (트레이드오프)
# ==================================================
def ask_priority():
    print("\n⚖️ 모든 조건을 동시에 만족하는 카드는 없습니다.")
    print("어떤 조건을 더 중요하게 생각하시나요?")
    print("1️⃣ 캐시백")
    print("2️⃣ 카페 할인")

    return "캐시백" if input("번호 선택: ") == "1" else "카페"


# ==================================================
# 8. RAG 검색 (카드 유형별)
# ==================================================
def retrieve_cards(query: str, card_type: str, k: int = 5) -> str:
    if card_type == "체크카드":
        docs = check_vectordb.similarity_search(query, k=k)

    # elif card_type == "신용카드":
    #     docs = credit_vectordb.similarity_search(query, k=k)

    else:
        docs = []

    return "\n\n".join(doc.page_content for doc in docs)


# ==================================================
# 9. 프롬프트
# ==================================================
def build_prompt(age, card_type, condition_text, positive, negative, coverage, retrieved_cards):
    coverage_text = "\n".join(
        [f"- {k}: {'충족됨' if v else '충족 불가'}" for k, v in coverage.items()]
    )

    return f"""
당신은 카드 상품 추천 금융 AI입니다.

아래 카드 정보에 포함된 내용만 사용하여
사용자의 조건에 가장 적합한 카드 TOP 3를 추천하세요.

[사용자 정보]
- 나이: {age}
- 카드 유형: {card_type}

[사용자가 입력한 요구사항]
{condition_text}

[중요 조건]
{", ".join(positive)}

[배제 조건]
{", ".join(negative) if negative else "없음"}

[요구사항 충족 여부]
{coverage_text}

[지시사항]
- 카드 정보에 없는 혜택은 절대 생성하지 마세요.
- 모든 조건을 만족하지 못할 경우,
  충족/미충족 조건을 명확히 설명하세요.

[출력 형식]

입력한 요구사항 요약:
- 주요 조건:
- 배제 조건:

TOP 3 카드 추천

1️⃣ 카드명:
- 카드사:
- 추천 이유:

2️⃣ 카드명:
- 카드사:
- 추천 이유:

3️⃣ 카드명:
- 카드사:
- 추천 이유:

[카드 정보]
{retrieved_cards}
"""


# ==================================================
# 10. 추천 실행
# ==================================================
def recommend_cards(age, card_type, condition_text):
    positive, negative = split_conditions(condition_text)
    retrieved_cards = retrieve_cards(condition_text, card_type)
    coverage = analyze_coverage(positive, retrieved_cards)

    if "캐시백" in positive and "카페" in positive and not coverage.get("캐시백", True):
        priority = ask_priority()
        retrieved_cards = retrieve_cards(f"{condition_text} {priority}", card_type)

    prompt = build_prompt(
        age, card_type, condition_text,
        positive, negative, coverage, retrieved_cards
    )

    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "system", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content.strip()


# ==================================================
# 11. CLI 실행
# ==================================================
if __name__ == "__main__":
    print("💳 카드 추천 챗봇\n")

    age = input("1️⃣ 나이를 입력하세요: ").strip()

    card_type = input(
        "2️⃣ 어떤 카드를 찾고 계신가요? (신용카드 / 체크카드): "
    ).strip()

    if card_type == "신용카드":
        print(
            "\nℹ️ 현재는 신용카드 데이터가 준비되지 않아\n"
            "체크카드 기준으로 추천을 진행합니다."
        )
        card_type = "체크카드"

    condition = input(
        "3️⃣ 조건을 적어주세요!:\n"
        "👉 "
    )

    print("\n==============================")
    print("🤖 카드 추천 결과")
    print("==============================")

    print(recommend_cards(age, card_type, condition))
    print("\n==============================")


💳 카드 추천 챗봇


🤖 카드 추천 결과
입력한 요구사항 요약:
- 주요 조건: 카페는, 필요해, 이용하고, 버스를, 할인은, 자주
- 배제 조건: 영화관, 안가., 근데

TOP 3 카드 추천

1️⃣ 카드명: 일상의 기쁨카드(체크)
- 카드사: IBK기업은행
- 추천 이유: 대중교통 이용 시 버스, 지하철 건당 100원 할인 혜택이 있어서 자주 이용하는 버스를 할인 받을 수 있습니다.

2️⃣ 카드명: 해피락스타 체크카드
- 카드사: KB국민카드
- 추천 이유: 카페 중 스타벅스에서 20% 환급할인 혜택이 있어서 자주 이용하는 카페에서 할인을 받을 수 있습니다.

3️⃣ 카드명: 스타플러스 체크카드
- 카드사: KB국민카드
- 추천 이유: 카페 중 스타벅스, 커피빈에서 10% 할인 혜택이 있어서 자주 이용하는 카페에서 할인을 받을 수 있습니다.하지만, 주요 조건 중 '카페는' 조건을 충족하지 못합니다.



# 페르소나 10개 적용 + 카드 분류

In [ ]:
import os
import json
import getpass  # 보안 입력을 위한 라이브러리 추가
import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI

# ==========================================
# 1. 초기 설정 및 보안 인증 (getpass 적용)
# ==========================================

# API 키를 터미널/콘솔에서 직접 입력받습니다. (화면에 표시되지 않음)
MY_API_KEY = getpass.getpass("OpenAI API key: ")
client = OpenAI(api_key=MY_API_KEY)

# 기본 경로 설정
BASE_DATA_PATH = "../data"
DATA_PATHS = {
    "체크": os.path.join(BASE_DATA_PATH, "check_cards_benefits"),
    "신용": os.path.join(BASE_DATA_PATH, "credit_cards_benefits")
}

ko_embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="jhgan/ko-sroberta-multitask"
)

# Vector DB 설정
chroma_client = chromadb.PersistentClient(path="../data/test/card_vector_db")

# 기존 컬렉션 삭제 후 재생성
try:
    chroma_client.delete_collection(name="card_separated_collection")
except:
    pass

collection = chroma_client.create_collection(
    name="card_separated_collection",
    embedding_function=ko_embedding_func
)

# ==========================================
# 2. 통합 인덱싱 함수 (신용/체크 구분)
# ==========================================
def indexing_all_cards():
    all_chunks_text = []
    all_metadatas = []
    all_ids = []
    id_counter = 0
    
    MAX_CHUNK_SIZE = 300
    OVERLAP_SIZE = 50

    for card_type, folder_path in DATA_PATHS.items():
        if not os.path.exists(folder_path):
            print(f"⚠️ 경로 없음: {folder_path} (건너뜁니다)")
            continue

        print(f"📂 [{card_type}카드] 데이터 인덱싱 중...")
        file_list = [f for f in os.listdir(folder_path) if f.endswith('.json')]

        for file_name in file_list:
            with open(os.path.join(folder_path, file_name), 'r', encoding='utf-8') as f:
                card = json.load(f)
                
                card_name = card.get("card_name", "")
                company = card.get("company", "")
                performance = card.get("performance", "")
                overseas = card.get("overseas", "")
                
                card_header = f"[{card_type}카드] 카드명: {card_name} | 카드사: {company} | 전월실적: {performance}"

                for benefit in card.get("benefit", []):
                    category = benefit.get("category", "")
                    content = benefit.get("content", "").strip()

                    start = 0
                    while True:
                        end = start + MAX_CHUNK_SIZE
                        sub = content[start:end]
                        if not sub: break

                        chunk_text = (
                            f"{card_header}\n"
                            f"혜택분류: {category}\n"
                            f"상세내용: {sub}\n"
                            f"해외사용여부: {overseas}"
                        )
                        
                        all_chunks_text.append(chunk_text)
                        all_metadatas.append({
                            "name": card_name, 
                            "company": company,
                            "card_type": card_type,
                            "category": category
                        })
                        all_ids.append(f"chunk_{id_counter}")
                        id_counter += 1

                        if end >= len(content): break
                        start += (MAX_CHUNK_SIZE - OVERLAP_SIZE)

    # DB 저장 (Batch 처리)
    batch_size = 100
    for i in range(0, len(all_chunks_text), batch_size):
        collection.add(
            ids=all_ids[i:i+batch_size],
            documents=all_chunks_text[i:i+batch_size],
            metadatas=all_metadatas[i:i+batch_size]
        )
    print(f"✅ 총 {len(all_chunks_text)}개의 데이터가 DB에 등록되었습니다.\n")

# ==========================================
# 3. 필터링 기반 추천 함수
# ==========================================
def get_model_response(query, persona_dict, target_card_type=None):
    persona_name = persona_dict['name']
    persona_traits = persona_dict['traits']
    
    search_query = f"{persona_traits} {query}"
    
    # ChromaDB 'where' 필터 적용
    if target_card_type in ["체크", "신용"]:
        results = collection.query(
            query_texts=[search_query], 
            n_results=30,
            where={"card_type": target_card_type}
        )
    else:
        results = collection.query(query_texts=[search_query], n_results=30)

    raw_documents = results['documents'][0]
    available_card_names = set()
    card_context_map = {}
    
    for doc in raw_documents:
        first_line = doc.split("\n")[0]
        card_title = first_line.split("카드명: ")[1].split("|")[0].strip()
        available_card_names.add(card_title)
        
        if card_title not in card_context_map:
            card_context_map[card_title] = []
        card_context_map[card_title].append(doc)

    verified_list_str = ", ".join(list(available_card_names))
    formatted_context = ""
    for title, contents in card_context_map.items():
        formatted_context += f"\n### 카드 데이터: {title} ###\n"
        formatted_context += "\n".join(contents) + "\n"

    system_prompt = f"""
    당신은 금융 상품 추천 전문가입니다. 
    반드시 [검증된 카드 리스트] 내의 {target_card_type if target_card_type else '전체'}카드만 추천하십시오.

    [검증된 카드 리스트]
    {verified_list_str}

    [상세 데이터 정보]
    {formatted_context}

    [사용자 페르소나]
    - 성함: {persona_name}
    - 특징: {persona_traits}

    ⛔ 규칙:
    1. 반드시 [검증된 카드 리스트]에 있는 카드만 추천하십시오.
    2. 추천 시 '체크카드'인지 '신용카드'인지 명확히 밝히십시오.
    3. 페르소나의 소비 패턴과 가장 일치하는 혜택을 상세히 설명하십시오.
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        temperature=0.0
    )
    return response.choices[0].message.content

# ==========================================
# 4. 실행 (10개 페르소나 시나리오)
# ==========================================

indexing_all_cards()

test_scenarios = [
    {"persona": {"name": "20대 자기계발생", "traits": "무실적 카드 선호, 대중교통 및 학원 이용 잦음, 편의점/카페 지출 위주"}, "query": "전월 실적 부담 없으면서 대중교통, 편의점, 학원 혜택이 좋은 체크카드 알려줘.", "target": "체크"},
    {"persona": {"name": "20대 사회초년생 자취형", "traits": "첫 직장, 배달·카페·온라인 쇼핑 비중 높음, 고정비 부담 큼"}, "query": "배달 음식과 카페, 온라인 쇼핑에서 자주 쓰기 좋은 체크카드 추천해줘.", "target": "체크"},
    {"persona": {"name": "20대 외출·소비 활동형", "traits": "외식과 카페 이용 빈도 높음, 문화생활도 가끔 즐김, 소액 결제 잦음"}, "query": "카페와 외식 위주로 자주 쓰기 좋고 체감 할인율이 높은 체크카드를 알려줘.", "target": "체크"},
    {"persona": {"name": "30대 출퇴근 직장인", "traits": "대중교통 출퇴근, 점심 외식과 커피 지출 반복, 소비 패턴 규칙적"}, "query": "출퇴근 교통비랑 회사 근처 카페, 점심 외식에 혜택이 집중된 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 자차 보유 직장인", "traits": "자차 출퇴근, 주유비 지출 큼, 주차·차량 유지비 사용"}, "query": "주유 혜택이 크고 자차 이용자에게 실질적으로 도움이 되는 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 여행·해외결제 중심형", "traits": "해외 여행 및 직구 빈번, 항공권·숙박·해외 결제 비중 높음"}, "query": "해외 결제 적립이나 수수료 혜택이 좋고 여행 시 활용하기 좋은 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "30대 건강관리 소비형", "traits": "병원, 약국, 헬스장 이용 빈도 높음, 생활비 관리도 중요"}, "query": "병원이나 헬스장, 건강관리 관련 지출에 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "40대 가족 생활형", "traits": "자녀 있음, 대형마트 장보기 잦음, 생활비와 고정비 관리 중요"}, "query": "대형마트 장보기랑 생활비 지출에 혜택이 잘 적용되는 신용카드를 추천해줘.", "target": "신용"},
    {"persona": {"name": "40대 교육비 집중형", "traits": "자녀 학원비 결제 비중 큼, 교육 관련 고정 지출 존재"}, "query": "자녀 학원비 결제 시 혜택을 받을 수 있는 신용카드를 알려줘.", "target": "신용"},
    {"persona": {"name": "디지털·구독 소비형", "traits": "OTT, 온라인 쇼핑, 디지털 구독 서비스 이용 빈도 높음"}, "query": "온라인 결제나 구독 서비스에서 자주 쓰기 좋고 할인이나 적립이 있는 체크카드 추천해줘.", "target": "체크"}
]

print("🚀 10개 페르소나 추천 테스트 시작\n")
for i, sc in enumerate(test_scenarios):
    print(f"📍 [시나리오 {i+1}] {sc['persona']['name']} ({sc['target']}카드 추천)")
    print(get_model_response(sc['query'], sc['persona'], target_card_type=sc['target']))
    print("\n" + "="*80 + "\n")


📂 [체크카드] 데이터 인덱싱 중...
📂 [신용카드] 데이터 인덱싱 중...
